# Lab 02 — Chapter 14: Core n-gram implementation

Notebook này tự cài đặt unigram, bigram và trigram language model.

## 1. Cài đặt tokenizer, corpus loader và N-gram Language Model

In [ ]:
from collections import Counter, defaultdict
from math import exp, inf, log
from pathlib import Path
from typing import Sequence
import json
import re

ORDERS = (1, 2, 3)
UNKNOWN_TOKEN = '<unk>'
TOKEN_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?")
SENTENCE_BOUNDARY_PATTERN = re.compile(r'[.!?]+(?:\s+|$)')

def tokenize(text: str) -> list[str]:
    """Chuẩn hóa text thành token chữ thường."""
    return TOKEN_PATTERN.findall(text.lower())

def load_sentences(dataset_path: Path, document_limit: int) -> tuple[int, list[list[str]]]:
    """Đọc JSONL và trả về số documents cùng các câu đã tokenize."""
    documents = 0
    sentences: list[list[str]] = []
    with dataset_path.open(encoding='utf-8') as dataset_file:
        for line in dataset_file:
            text = json.loads(line).get('text', '')
            if not isinstance(text, str) or not text.strip():
                continue
            pieces = SENTENCE_BOUNDARY_PATTERN.split(text)
            sentences.extend(tokens for tokens in map(tokenize, pieces) if tokens)
            documents += 1
            if documents >= document_limit:
                break
    if not sentences:
        raise ValueError('Corpus không có câu hợp lệ.')
    return documents, sentences

class NGramLanguageModel:
    """Language model n-gram tự cài đặt, hỗ trợ MLE và Laplace smoothing."""

    def __init__(self, order: int, smoothing: bool = False) -> None:
        """Khởi tạo model theo order và tùy chọn smoothing boolean."""
        if order not in ORDERS:
            raise ValueError(f'Order không hợp lệ: {order}.')
        self.order = order
        self.smoothing = smoothing
        self.vocabulary: set[str] = set()
        self.context_counts: Counter[tuple[str, ...]] = Counter()
        self.continuation_counts: dict[tuple[str, ...], Counter[str]] = defaultdict(Counter)

    def fit(self, corpus: Sequence[Sequence[str]]) -> 'NGramLanguageModel':
        """Huấn luyện model bằng count các context và từ tiếp theo."""
        self.vocabulary = {token for sentence in corpus for token in sentence}
        if not self.vocabulary:
            raise ValueError('Training corpus không có token.')
        self.vocabulary.add(UNKNOWN_TOKEN)
        for sentence in corpus:
            for position, word in enumerate(sentence):
                for context_size in range(min(self.order - 1, position) + 1):
                    context = tuple(sentence[position - context_size:position])
                    self.context_counts[context] += 1
                    self.continuation_counts[context][word] += 1
        return self

    def probability(self, context: Sequence[str], word: str) -> float:
        """Trả về xác suất điều kiện P(word | context)."""
        maximum_context = self.order - 1
        suffix = list(context)[-maximum_context:] if maximum_context else []
        normalized_context = tuple(token if token in self.vocabulary else UNKNOWN_TOKEN for token in suffix)
        normalized_word = word if word in self.vocabulary else UNKNOWN_TOKEN
        context_count = self.context_counts[normalized_context]
        word_count = self.continuation_counts[normalized_context][normalized_word]
        if self.smoothing:
            return (word_count + 1) / (context_count + len(self.vocabulary))
        return word_count / context_count if context_count else 0.0

    def sentence_log_probability(self, sentence: Sequence[str]) -> float:
        """Tính log xác suất câu để tránh numerical underflow."""
        score = 0.0
        for position, word in enumerate(sentence):
            context = sentence[max(0, position - self.order + 1):position]
            probability = self.probability(context, word)
            if probability == 0.0:
                return float('-inf')
            score += log(probability)
        return score

    def sentence_probability(self, sentence: Sequence[str]) -> float:
        """Tính xác suất câu từ log probability ổn định về số học."""
        score = self.sentence_log_probability(sentence)
        return 0.0 if score == float('-inf') else exp(score)

    def perplexity(self, corpus: Sequence[Sequence[str]]) -> float:
        """Tính perplexity trung bình theo token của một evaluation corpus."""
        token_count = sum(len(sentence) for sentence in corpus)
        if token_count == 0:
            raise ValueError('Evaluation corpus không có token.')
        score = sum(self.sentence_log_probability(sentence) for sentence in corpus)
        return inf if score == float('-inf') else exp(-score / token_count)


## 2. Kiểm chứng trên corpus nhỏ

In [ ]:
training_corpus = [
    tokenize('the cat eats fish'),
    tokenize('the cat likes fish'),
    tokenize('the dog eats meat'),
]
bigram_model = NGramLanguageModel(order=2).fit(training_corpus)
trigram_model = NGramLanguageModel(order=3).fit(training_corpus)
verification = {
    'P(cat|the)': bigram_model.probability(('the',), 'cat'),
    'P(eats|cat)': bigram_model.probability(('cat',), 'eats'),
    'P(eats|the cat)': trigram_model.probability(('the', 'cat'), 'eats'),
    'P(the cat eats fish)': bigram_model.sentence_probability(tokenize('the cat eats fish')),
}
display(verification)
